In [59]:
! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community langchain_openai

Using Python 3.12.13 environment at: /home/arun/Work/End-to-End-Advanced-RAG-Project-with-LLMOPS-/.venv
Checked 7 packages in 6ms


In [60]:
import os 

from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

## Data Ingestion

In [61]:
from langchain_community.document_loaders import TextLoader

In [62]:
loader = TextLoader("../data/agentic_ai.txt", encoding="utf-8")
docs = loader.load()

In [63]:
docs[0].page_content[:500]

'# Comprehensive Guide to Agentic AI and Its Real-World Applications\n\n## 1. Introduction to Agentic AI\nAgentic AI refers to artificial intelligence systems capable of autonomous decision-making, goal setting, reasoning, and executing multi-step actions in dynamic environments with minimal human intervention. Unlike traditional passive Large Language Models (LLMs) that merely answer user queries in single turns, agentic systems proactively perceive their environment, devise plans, leverage externa'

In [64]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [65]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50,separators=["\n\n", "\n", " ", ""])

In [66]:
chunks = text_splitter.split_documents(docs)

In [67]:
print(f"Total chunks created : {len(chunks)}")
print(f"First chunks:\n{chunks[0].page_content}")

Total chunks created : 19
First chunks:
# Comprehensive Guide to Agentic AI and Its Real-World Applications


In [68]:
for i in range(0,19):
 print(f"{i} chunks:\n{chunks[i].page_content}")


0 chunks:
# Comprehensive Guide to Agentic AI and Its Real-World Applications
1 chunks:
## 1. Introduction to Agentic AI
2 chunks:
Agentic AI refers to artificial intelligence systems capable of autonomous decision-making, goal setting, reasoning, and executing multi-step actions in dynamic environments with minimal human intervention. Unlike traditional passive Large Language Models (LLMs) that merely answer user queries in single turns, agentic systems proactively perceive their environment, devise plans, leverage external tools, inspect feedback, iterate on mistakes, and collaborate with other agents to accomplish
3 chunks:
and collaborate with other agents to accomplish complex objectives.
4 chunks:
## 2. Core Architectural Pillars of Agentic AI
Modern agentic architectures typically consist of four primary pillars:

### A. Perception and Goal Formulation
Agents interpret multi-modal inputs, including user instructions, environmental state observations, and real-time streaming data

In [69]:
from dotenv import load_dotenv, find_dotenv
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# Load environment variables
load_dotenv(find_dotenv())

# 1. Initialize FastEmbed (Runs locally, completely free and fast)
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# 2. Build In-Memory Vector Store from chunks
vector_store = InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings
)

print(f"Indexed {len(chunks)} chunks into vector store using FastEmbed!")

Indexed 19 chunks into vector store using FastEmbed!


In [70]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [71]:
response = llm.invoke("Good morning")

print(response.content)

Good morning! ☀️ How can I help you today?


In [72]:
# List of dummy questions to test retrieval
dummy_questions = [
    "What are the core architectural pillars of Agentic AI?",
    "How is Agentic AI applied in software engineering?",
    "What is the difference between short-term and long-term memory in agents?",
    "What are the challenges of deploying agentic systems in production?"
]

# Run similarity search
for query in dummy_questions:
    print("=" * 60)
    print(f"Question: {query}")
    print("=" * 60)
    
    # Retrieve top 2 most relevant chunks
    results = vector_store.similarity_search(query, k=2)
    
    for i, doc in enumerate(results):
        print(f"\n[Chunk {i+1}]:")
        print(doc.page_content.strip())
    print("\n")

Question: What are the core architectural pillars of Agentic AI?

[Chunk 1]:
## 2. Core Architectural Pillars of Agentic AI
Modern agentic architectures typically consist of four primary pillars:

### A. Perception and Goal Formulation
Agents interpret multi-modal inputs, including user instructions, environmental state observations, and real-time streaming data. The agent extracts high-level objectives, decomposes ambiguous requirements into actionable tasks, and defines measurable success criteria.

[Chunk 2]:
## 1. Introduction to Agentic AI


Question: How is Agentic AI applied in software engineering?

[Chunk 1]:
# Comprehensive Guide to Agentic AI and Its Real-World Applications

[Chunk 2]:
## 4. Key Real-World Applications of Agentic AI

### A. Autonomous Software Engineering
AI agents can read entire codebases, reproduce bug reports, synthesize unit tests, generate pull requests, and review code diffs. Examples include Devin, Claude Code, and Copilot Workspace, which manage end

In [74]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Create a retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 2. Define the RAG prompt
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the provided context below. 
If you don't know the answer from the context, say that you don't know.

Context:
{context}

Question:
{question}

Answer:"""
)

# Helper function to format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3. Assemble LCEL chain: Retrieve -> Format -> Prompt -> LLM -> Parse
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Test with a question!
query = "Core Architectural Pillars of Agentic AI"
response = rag_chain.invoke(query)

print(f"Question: {query}\n")
print(f"Answer:\n{response}")

Question: Core Architectural Pillars of Agentic AI

Answer:
**Core Architectural Pillars of Agentic AI**

Based on the provided context, the primary pillar highlighted is:

1. **Perception and Goal Formulation** – Agents interpret multi‑modal inputs, extract high‑level objectives, decompose ambiguous requirements into actionable tasks, and define measurable success criteria.
